# LLM-DHRP: Language-Informed Differentiable Hierarchical Risk Parity

**Full Experimental Pipeline v2 (Colab T4 GPU)**

Clones the repo from GitHub, installs deps, and runs the full pipeline:
volume features, FF5+Momentum, expanded headlines, Transformer/PPO baselines,
BCa bootstrap CIs, FDR correction, sub-period analysis, TC sensitivity.

In [ ]:
# === CELL 1: SETUP ===
import os, sys
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

IN_COLAB = "google.colab" in sys.modules
REPO = "dhrp-allocation"
REPO_URL = "https://github.com/joseamador0898/dhrp-allocation.git"

if IN_COLAB:
    # Clone repo (or pull if already cloned)
    if not os.path.exists(f"/content/{REPO}"):
        os.system(f"git clone {REPO_URL} /content/{REPO}")
    else:
        os.system(f"cd /content/{REPO} && git pull")

    os.chdir(f"/content/{REPO}")
    sys.path.insert(0, f"/content/{REPO}")
    print(f"Working dir: {os.getcwd()}")

    # Only install what Colab doesn't have (torch/transformers are pre-installed)
    os.system("pip install -q fredapi python-dotenv feedparser datasets pandas-datareader cvxpy 2>/dev/null")

    # Upload .env if needed (contains FRED API key etc.)
    env_path = f"/content/{REPO}/.env"
    if not os.path.exists(env_path):
        print("\\n>>> Upload your .env file (contains FRED_API_KEY etc.) <<<")
        from google.colab import files
        uploaded = files.upload()
        for name, data in uploaded.items():
            with open(env_path, "wb") as f:
                f.write(data)
        print(f".env saved to {env_path}")
else:
    sys.path.insert(0, "..")

import warnings; warnings.filterwarnings("ignore")
import torch, numpy as np

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
device = "cuda" if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Device: {device}")

In [ ]:
# === CELL 2: LOAD PRICE DATA (all 3 universes) ===
from datetime import datetime, timedelta
from src.data.price_loader import load_universe, load_fama_french, UNIVERSES

END = datetime.now().strftime('%Y-%m-%d')
START = (datetime.now() - timedelta(days=10*365)).strftime('%Y-%m-%d')
print(f'Period: {START} to {END}\n')

print(f'=== DM Universe ({len(UNIVERSES["DM"])} ETFs) ===')
DM_prices = load_universe('DM', START, END)

print(f'\n=== EM Universe ({len(UNIVERSES["EM"])} ETFs) ===')
EM_prices = load_universe('EM', START, END)

print(f'\n=== Commodities Universe ({len(UNIVERSES["Commodities"])} ETFs) ===')
CMD_prices = load_universe('Commodities', START, END)

print('\n=== Fama-French Factors ===')
FF = load_fama_french(START, END)

In [ ]:
# === CELL 3: LOAD HEADLINES (Yahoo Finance) ===
from src.data.text_loader import load_yfinance_news, aggregate_headlines_for_rebalance
import pandas as pd

# Collect headlines for all tickers across universes
all_tickers = (
    list(UNIVERSES['DM'].values()) +
    list(UNIVERSES['EM'].values()) +
    list(UNIVERSES['Commodities'].values())
)
all_tickers = list(set(all_tickers))  # deduplicate

print(f'Fetching headlines for {len(all_tickers)} tickers...')
headlines_df = load_yfinance_news(all_tickers, START, END, max_headlines=100)
print(f'Collected {len(headlines_df)} headlines')
if not headlines_df.empty:
    print(f'Date range: {headlines_df["date"].min()} to {headlines_df["date"].max()}')
    print(f'Tickers with news: {headlines_df["ticker"].nunique()}')
    print(headlines_df.groupby('ticker').size().describe())

In [ ]:
# === CELL 4: FINBERT EMBEDDINGS (T4: ~2 min for 1000 headlines) ===
from src.data.llm_features import get_finbert_embeddings

if not headlines_df.empty:
    unique_headlines = headlines_df['headline'].unique().tolist()
    print(f'Extracting FinBERT embeddings for {len(unique_headlines)} unique headlines...')
    
    finbert_embs = get_finbert_embeddings(unique_headlines, batch_size=64, device=device)
    print(f'FinBERT embeddings shape: {finbert_embs.shape}')
    
    # Create mapping: headline -> embedding
    headline_to_emb = {h: finbert_embs[i] for i, h in enumerate(unique_headlines)}
    print(f'VRAM after FinBERT: {torch.cuda.memory_allocated()/1e9:.2f} GB' if torch.cuda.is_available() else 'CPU mode')
else:
    print('No headlines available. Proceeding with price-only features.')
    headline_to_emb = {}

In [ ]:
# === CELL 5: QWEN3-8B SENTIMENT (T4: ~5 min for 100 batches) ===
from src.data.llm_features import get_qwen3_sentiment, sentiment_to_features

# Group headlines by ticker for batch sentiment extraction
if not headlines_df.empty and torch.cuda.is_available():
    # Group headlines by ticker, take top 10 per ticker
    ticker_groups = headlines_df.groupby('ticker')['headline'].apply(
        lambda x: x.head(10).tolist()
    ).to_dict()
    
    print(f'Extracting Qwen3-8B sentiment for {len(ticker_groups)} tickers...')
    batches = list(ticker_groups.values())
    
    # Process in chunks to manage VRAM
    CHUNK = 20
    all_sentiments = []
    for i in range(0, len(batches), CHUNK):
        chunk = batches[i:i+CHUNK]
        print(f'  Chunk {i//CHUNK + 1}/{(len(batches)+CHUNK-1)//CHUNK}...')
        sents = get_qwen3_sentiment(chunk, device=device)
        all_sentiments.extend(sents)
    
    # Map ticker -> sentiment features
    ticker_sentiment = {}
    for ticker, sent in zip(ticker_groups.keys(), all_sentiments):
        ticker_sentiment[ticker] = sentiment_to_features(sent)
    
    print(f'Sentiment extracted for {len(ticker_sentiment)} tickers')
    print(f'VRAM after Qwen3: {torch.cuda.memory_allocated()/1e9:.2f} GB')
else:
    print('Skipping Qwen3 sentiment (no GPU or no headlines)')
    ticker_sentiment = {}

In [ ]:
# === CELL 6: BUILD TEXT FEATURE TENSORS ===
from src.data.feature_engineering import build_dataset

def build_text_tensor_for_universe(prices, universe_tickers, headline_to_emb, headlines_df):
    """Build (n_samples, n_assets, 768) FinBERT tensor for a universe."""
    X, S, R, H = build_dataset(prices)
    n_samp = X.shape[0]
    n_assets = prices.shape[1]
    ticker_list = list(universe_tickers.values())
    
    # For each asset, use the average embedding of all its headlines
    asset_embs = np.zeros((n_assets, 768), dtype=np.float32)
    for j, ticker in enumerate(ticker_list):
        ticker_headlines = headlines_df[headlines_df['ticker'] == ticker]['headline'].tolist()
        embs = [headline_to_emb[h] for h in ticker_headlines if h in headline_to_emb]
        if embs:
            asset_embs[j] = np.mean(embs, axis=0)
    
    # Broadcast same embedding across all time steps (static text features)
    # In production, you'd have time-varying headlines per rebalance window
    text_tensor = np.tile(asset_embs, (n_samp, 1, 1))  # (n_samp, n_assets, 768)
    return text_tensor

# Build text features for each universe
text_dm, text_em, text_cmd = None, None, None

if headline_to_emb:
    print('Building text feature tensors...')
    text_dm = build_text_tensor_for_universe(DM_prices, UNIVERSES['DM'], headline_to_emb, headlines_df)
    text_em = build_text_tensor_for_universe(EM_prices, UNIVERSES['EM'], headline_to_emb, headlines_df)
    text_cmd = build_text_tensor_for_universe(CMD_prices, UNIVERSES['Commodities'], headline_to_emb, headlines_df)
    print(f'  DM text: {text_dm.shape}')
    print(f'  EM text: {text_em.shape}')
    print(f'  CMD text: {text_cmd.shape}')
    
    # Save for reuse
    os.makedirs('results/features', exist_ok=True)
    np.savez_compressed('results/features/text_dm.npz', finbert=text_dm)
    np.savez_compressed('results/features/text_em.npz', finbert=text_em)
    np.savez_compressed('results/features/text_cmd.npz', finbert=text_cmd)
    print('Saved to results/features/')
else:
    print('No text features available. Using price-only mode.')

In [ ]:
# === CELL 7: LOAD MACRO FEATURES (FRED) ===
from src.data.fred_loader import load_fred_data, make_macro_features
from dotenv import load_dotenv
load_dotenv()

print('Loading FRED macro features...')
fred_df = load_fred_data(START, END)
if not fred_df.empty:
    macro_df = make_macro_features(fred_df)
    print(f'Macro features: {macro_df.shape}')
    print(f'Columns: {macro_df.columns.tolist()}')
    macro_df.tail()
else:
    macro_df = None
    print('FRED not available. Proceeding without macro features.')

In [ ]:
# === CELL 8: TRAIN ALL MODELS (DM Universe) ===
from src.training.trainer import train_dhrp, train_llm_dhrp

print('=== DEVELOPED MARKETS ===')

# A1: Price-only DHRP (baseline)
print('\n--- A1: Training DHRP (price-only) ---')
dhrp_dm = train_dhrp(DM_prices, device=device, is_em=False)

# A4: Full LLM-DHRP (price + FinBERT + covariance)
llm_dhrp_dm = None
if text_dm is not None:
    print('\n--- A4: Training LLM-DHRP (price + text) ---')
    llm_dhrp_dm = train_llm_dhrp(
        DM_prices,
        text_features={'finbert': text_dm},
        macro_features=macro_df.values if macro_df is not None else None,
        device=device, is_em=False,
        use_text=True, use_macro=macro_df is not None,
        fusion_type='cross_attention', depth=3,
        epochs=60, lr=3e-4,
    )

# Save models
os.makedirs('results/models', exist_ok=True)
torch.save(dhrp_dm.state_dict(), 'results/models/dhrp_dm.pt')
if llm_dhrp_dm is not None:
    torch.save(llm_dhrp_dm.state_dict(), 'results/models/llm_dhrp_dm.pt')
print('\nModels saved.')

In [ ]:
# === CELL 9: TRAIN EM + COMMODITIES ===

print('=== EMERGING MARKETS ===')
print('\n--- Training DHRP (EM) ---')
dhrp_em = train_dhrp(EM_prices, device=device, is_em=True)

llm_dhrp_em = None
if text_em is not None:
    print('\n--- Training LLM-DHRP (EM) ---')
    llm_dhrp_em = train_llm_dhrp(
        EM_prices, text_features={'finbert': text_em},
        device=device, is_em=True,
        epochs=50, lr=1.5e-4,
    )

print('\n=== COMMODITIES ===')
print('\n--- Training DHRP (Commodities) ---')
dhrp_cmd = train_dhrp(CMD_prices, device=device, is_em=False)

llm_dhrp_cmd = None
if text_cmd is not None:
    print('\n--- Training LLM-DHRP (Commodities) ---')
    llm_dhrp_cmd = train_llm_dhrp(
        CMD_prices, text_features={'finbert': text_cmd},
        device=device, is_em=False,
        epochs=60, lr=3e-4,
    )

# Save all models
torch.save(dhrp_em.state_dict(), 'results/models/dhrp_em.pt')
torch.save(dhrp_cmd.state_dict(), 'results/models/dhrp_cmd.pt')
if llm_dhrp_em: torch.save(llm_dhrp_em.state_dict(), 'results/models/llm_dhrp_em.pt')
if llm_dhrp_cmd: torch.save(llm_dhrp_cmd.state_dict(), 'results/models/llm_dhrp_cmd.pt')
print('\nAll models saved.')

In [ ]:
# === CELL 10: ROLLING BACKTEST (all universes) ===
from src.evaluation.backtest import rolling_backtest

METHODS = ['EW', 'MINVAR', 'MV', 'HRP', 'RP', 'MAXDIV', 'DHRP']
if llm_dhrp_dm is not None:
    METHODS.append('LLM_DHRP')

print('Running backtests...\n')

dm_res = rolling_backtest(
    DM_prices, is_em=False, dhrp_model=dhrp_dm,
    llm_dhrp_model=llm_dhrp_dm, text_features={'finbert': text_dm} if text_dm is not None else None,
    macro_features=macro_df, methods=METHODS,
)
print(f'DM: {len(dm_res)} observations')

em_res = rolling_backtest(
    EM_prices, is_em=True, dhrp_model=dhrp_em,
    llm_dhrp_model=llm_dhrp_em, text_features={'finbert': text_em} if text_em is not None else None,
    methods=METHODS,
)
print(f'EM: {len(em_res)} observations')

cmd_res = rolling_backtest(
    CMD_prices, is_em=False, dhrp_model=dhrp_cmd,
    llm_dhrp_model=llm_dhrp_cmd, text_features={'finbert': text_cmd} if text_cmd is not None else None,
    methods=METHODS,
)
print(f'Commodities: {len(cmd_res)} observations')

In [ ]:
# === CELL 11: RESULTS & STATISTICAL TESTS ===
from src.evaluation.statistics import compute_stats, sharpe_difference_test, diebold_mariano_test
from src.evaluation.factor_analysis import factor_analysis
import pandas as pd

for label, res, prices, is_em in [
    ('DM', dm_res, DM_prices, False),
    ('EM', em_res, EM_prices, True),
    ('Commodities', cmd_res, CMD_prices, False),
]:
    print(f'\n{"="*60}')
    print(f'  {label} RESULTS ({prices.shape[1]} assets)')
    print(f'{"="*60}')
    
    stats = compute_stats(res)
    try:
        factors = factor_analysis(res, FF)
        table = stats.merge(factors, on='Method', how='left').round(3)
    except Exception:
        table = stats.round(3)
    print(table.to_string(index=False))
    
    # Statistical tests vs HRP
    print(f'\n--- Statistical Tests (vs HRP) ---')
    for m in sorted(res['method'].unique()):
        if m == 'HRP':
            continue
        try:
            diff = sharpe_difference_test(res, m, 'HRP')
            dm_test = diebold_mariano_test(res, m, 'HRP')
            sig = ' ***' if diff['bootstrap_p'] < 0.01 else ' **' if diff['bootstrap_p'] < 0.05 else ' *' if diff['bootstrap_p'] < 0.10 else ''
            print(f"  {m:12s} vs HRP: Sharpe diff={diff['sharpe_diff']:+.3f} "
                  f"[{diff['bootstrap_ci_lo']:.3f}, {diff['bootstrap_ci_hi']:.3f}] "
                  f"p={diff['bootstrap_p']:.3f}{sig} | "
                  f"DM stat={dm_test['DM_stat']:+.3f} p={dm_test['p_value']:.3f}")
        except Exception as e:
            print(f'  {m:12s} vs HRP: failed ({e})')
    
    # Save results
    table.to_csv(f'results/{label}_results.csv', index=False)

In [ ]:
# === CELL 12: PAPER FIGURES ===
from src.visualization.plots import plot_cumulative, plot_sharpe_bars
import matplotlib.pyplot as plt

os.makedirs('results/figures', exist_ok=True)

results_dict = {'DM': dm_res, 'EM': em_res, 'Commodities': cmd_res}
plot_cumulative(results_dict, output_dir='results/figures')
plot_sharpe_bars(results_dict, output_dir='results/figures')
print('Cumulative returns and Sharpe bar figures saved.')

# LLM-DHRP vs DHRP delta analysis
if 'LLM_DHRP' in dm_res['method'].unique():
    from src.visualization.plots import get_series
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    for ax, (uname, res) in zip(axes, results_dict.items()):
        s_llm = get_series(res, 'LLM_DHRP')
        s_dhrp = get_series(res, 'DHRP')
        common = s_llm.index.intersection(s_dhrp.index)
        diff = s_llm.loc[common] - s_dhrp.loc[common]
        cum_diff = diff.cumsum() * 100
        ax.fill_between(common, cum_diff.values, 0,
                        where=cum_diff.values >= 0, alpha=0.3, color='green')
        ax.fill_between(common, cum_diff.values, 0,
                        where=cum_diff.values < 0, alpha=0.3, color='red')
        ax.plot(common, cum_diff.values, color='black', lw=1.5)
        ax.axhline(0, color='black', ls='--', lw=0.8)
        ax.set_title(f'{uname}: LLM-DHRP minus DHRP', fontweight='bold')
        ax.set_ylabel('Cumulative Excess Return (%)')
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('results/figures/llm_delta.png', dpi=300)
    plt.show()
    print('LLM delta figure saved.')

In [ ]:
# === CELL 13: GATING INTERPRETABILITY ===
# Visualize how text features change the tree routing decisions
import matplotlib.pyplot as plt
import seaborn as sns

if llm_dhrp_dm is not None:
    from src.data.feature_engineering import build_dataset, make_features
    
    X, S, R, H = build_dataset(DM_prices)
    n_samples = min(50, X.shape[0])
    
    # Compare gating with vs without text
    probs_with_text = []
    probs_without_text = []
    
    for i in range(n_samples):
        x = torch.from_numpy(X[i]).to(device)
        s = torch.from_numpy(S[i]).to(device)
        
        if text_dm is not None:
            te = torch.from_numpy(text_dm[i].mean(axis=0)).to(device)
        else:
            te = torch.randn(768).to(device)  # random for demo
        
        p_with = llm_dhrp_dm.get_gating_probs(x, s, text_emb=te)
        p_without = llm_dhrp_dm.get_gating_probs(x, s, text_emb=None)
        probs_with_text.append([p.cpu().numpy() for p in p_with])
        probs_without_text.append([p.cpu().numpy() for p in p_without])
    
    # Plot root node gating shift
    root_with = [p[0][0] for p in probs_with_text]  # P(left) at root
    root_without = [p[0][0] for p in probs_without_text]
    
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(root_without, label='Price-only', alpha=0.7, color='steelblue')
    ax.plot(root_with, label='Price + Text', alpha=0.7, color='crimson')
    ax.fill_between(range(n_samples),
                    [a-b for a, b in zip(root_with, root_without)],
                    alpha=0.2, color='crimson', label='Text impact')
    ax.set_xlabel('Sample')
    ax.set_ylabel('P(left) at root node')
    ax.set_title('Root Node Gating: Impact of Text Features', fontweight='bold')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('results/figures/gating_interpretability.png', dpi=300)
    plt.show()
    print('Gating interpretability figure saved.')
else:
    print('LLM-DHRP not trained. Skipping interpretability analysis.')

In [ ]:
# === CELL 14: FINAL SUMMARY ===
print('\n' + '='*70)
print('  EXPERIMENT COMPLETE')
print('='*70)

print(f'\nUniverses tested: DM ({DM_prices.shape[1]}), EM ({EM_prices.shape[1]}), CMD ({CMD_prices.shape[1]})')
print(f'Total assets: {DM_prices.shape[1] + EM_prices.shape[1] + CMD_prices.shape[1]}')
print(f'Methods compared: {sorted(dm_res["method"].unique())}')
print(f'Device used: {device}')

print('\nKey files saved:')
print('  results/DM_results.csv')
print('  results/EM_results.csv')
print('  results/Commodities_results.csv')
print('  results/models/*.pt')
print('  results/features/*.npz')
print('  results/figures/*.png')

if torch.cuda.is_available():
    print(f'\nGPU memory peak: {torch.cuda.max_memory_allocated()/1e9:.2f} GB')